## Hyperparameter Tuning : Optuna

In [7]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import optuna


In [8]:
X, y = make_classification(n_samples=1000, n_features=20, n_classes=2, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [9]:
X_train, y_train = torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.long)
X_test, y_test = torch.tensor(X_test, dtype=torch.float32), torch.tensor(y_test, dtype=torch.long)

In [10]:
class SimpleNN(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 2)
        )

    def forward(self, x):
        return self.network(x)

In [11]:
def objective(trial):

    learning_rate = trial.suggest_loguniform('learning_rate', 1e-4, 1e-1)
    hidden_dim = trial.suggest_int('hidden_dim', 16, 128)

    model = SimpleNN(input_dim=20, hidden_dim=hidden_dim)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    epochs = 20
    batch_size = 32

    train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=batch_size, shuffle=True)

    for epoch in range(epochs):
        model.train()
        for batch_X, batch_y in train_loader:
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()


    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for batch_X, batch_y in test_loader:
            outputs = model(batch_X)
            _, predicted = torch.max(outputs, 1)
            total += batch_y.size(0)
            correct = (predicted == batch_y).sum().item()

    accuracy = correct / total
    return accuracy


study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20)

[I 2026-10-02 01:19:14,085] A new study created in memory with name: no-name-b55d0e53-64d5-4e7a-ac6e-db56533b6c0a
C:\Users\alili\AppData\Local\Temp\ipykernel_23164\1970177792.py:3: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  learning_rate = trial.suggest_loguniform('learning_rate', 1e-4, 1e-1)
[I 2026-10-02 01:19:17,367] Trial 0 finished with value: 0.03 and parameters: {'learning_rate': 0.0001139546030490758, 'hidden_dim': 128}. Best is trial 0 with value: 0.03.
[I 2026-10-02 01:19:19,291] Trial 1 finished with value: 0.03 and parameters: {'learning_rate': 0.005087435824943788, 'hidden_dim': 35}. Best is trial 0 with value: 0.03.
[I 2026-10-02 01:19:20,917] Trial 2 finished with value: 0.03 and parameters: {'learning_rate': 0.000100601317105586, 'hidden_dim': 37}. Best is trial 0 with value: 0.03.
[I 2026-10-02 01:19:22,678]

In [12]:
study.best_params

{'learning_rate': 0.0030612859123576707, 'hidden_dim': 69}